In [1]:
import os
from pathlib import Path
os.chdir(path = Path(r"C:\Users\apaks\projects\YT-RAG"))

In [11]:
from src.yt_rag.components.search import RAGSearch
import json

from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval import evaluate
from deepeval.evaluate.configs import CacheConfig


### Faitfulness

In [3]:
GOLDENS_PATH = "rag_evaluation/evaluation_set/deep_eval_goldens"
JUDGE_MODEL = "gpt-4o-mini"
THRESHOLD = 0.7

In [ ]:
files = os.listdir(GOLDENS_PATH)

In [ ]:
files

In [4]:
test_cases = []
file_path = Path(GOLDENS_PATH, '0jspaMLxBig.json')
with open(file_path, "r") as f:
    golden_dataset = json.load(f)

generator = RAGSearch(url="https://www.youtube.com/watch?v=0jspaMLxBig&t=99s")

for g in golden_dataset['goldens']:
    context = g['context']
    answer = generator.generate_response(context = context, query = g['input'])

    test_cases.append(
        LLMTestCase(
            input = g['input'],
            actual_output = answer,
            retrieval_context=context,
            # no expected output
        )
    )
    

In [5]:
golden_dataset

{'video_title': 'Andrew Ng: Deep Learning, Education, and Real-World AI | Lex Fridman Podcast #73',
 'video_url': 'https://www.youtube.com/watch?v=0jspaMLxBig&t=99s',
 'goldens': [{'input': 'What was the primary reason Andrew Ng decided to create online courses?',
   'expected_output': 'His goal was to help anyone with an interest in machine learning to break into the field and foster a broad, global community of learners.',
   'context': ['broad as possible as global as possibleso really try to reach as many peopleinterested in machine learning and AI aspossible I really want to help anyonethat had an interest in machine learningto break into fields'],
   'metadata': {'timestamp': '08:00', 'difficulty': 'easy'}},
  {'input': "What was the 'failed' feature Andrew Ng attempted to implement in his early online education platform?",
   'expected_output': 'He built a feature that allowed multiple people to be logged into the same website simultaneously to watch videos together, but it was 

In [6]:
test_cases

[LLMTestCase(input='What was the primary reason Andrew Ng decided to create online courses?', actual_output='Final Answer: The primary reason Andrew Ng decided to create online courses was to help anyone interested in machine learning break into the field and to reach as many people as possible.', expected_output=None, context=None, retrieval_context=['broad as possible as global as possibleso really try to reach as many peopleinterested in machine learning and AI aspossible I really want to help anyonethat had an interest in machine learningto break into fields'], metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, flaky=False, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input="What was the 'failed' feature Andrew Ng attempted to implement in his early online education platform?", actual_output="F

In [7]:
metrics = [FaithfulnessMetric(
    threshold= THRESHOLD, 
    model = JUDGE_MODEL,
    include_reason = True),
AnswerRelevancyMetric(
    threshold= THRESHOLD,
    model = JUDGE_MODEL,
    include_reason= True) 
]

In [16]:
results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    cache_config=CacheConfig(
        use_cache=False,
        write_cache=False
    )
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

c:\Users\apaks\projects\YT-RAG\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\apaks\AppData\Local\Programs\Python\Python313\Lib\inspect.py:3061: RuntimeWarning: coroutine 
'Kernel.shell_main' was never awaited
  params = OrderedDict((param.name, param) for param in parameters)
RuntimeWarning: Enable tracemalloc to get the object allocation traceback

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            What was the significance of the helicopter project in Pieter Abbeelâ€™s PhD           │
│  │                       thesis?                                                                                │
│  │     Actual Output:    The retrieved context does not contain enough information to answer this.              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness     │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...        │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.70      │ The score is 0.00 because the output failed to address    │
│              │                  │       │           │ the significance of the helicopter project, instead       │
│              │                  │       │           │ providing irrelevant information that did not relate to   │
│              │                  │       │           │ the input question.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:            According to Andrew Ng, what did researchers 'get wrong' during the early days of      │
│  │                       Google Brain?                                                                          │
│  │     Actual Output:    Final Answer: According to Andrew Ng, researchers 'got wrong' the belief that most     │
│  │                       knowledge was acquired through supervised learning, underestimating the power of       │
│  │                       unsupervised learning.                                                                 │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reaso

⚠ WARNING: No hyperparameters logged.
» ]8;id=5771142;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 56.35s | token cost: 0.0068958 USD)
» Test Results (10 total tests):
   » Pass Rate: 40.0% | Passed: 4 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [15]:
results

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions present, indicating that the actual output aligns perfectly with the retrieval context.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00032235, input_tokens=1533, output_tokens=154, verbose_logs='Truths (limit=None):\n[\n    "The speaker aims to reach a wide audience interested in machine learning and AI.",\n    "The speaker wants to help anyone interested in machine learning to enter the field."\n] \n \nClaims:\n[\n    "Andrew Ng created online courses to help anyone interested in machine learning break into the field.",\n    "Andrew Ng aimed to reach as many people as possible with his online courses."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": 